In [42]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [43]:
df = pd.read_csv('diabetes_prediction_dataset.csv')
df.head()

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [44]:
df.shape

(100000, 9)

In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  object 
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  object 
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 6.9+ MB


In [46]:
df['gender'].value_counts()

gender
Female    58552
Male      41430
Other        18
Name: count, dtype: int64

In [47]:
df['gender'] = df['gender'].map({'Male':1 , 'Female':0, 'Other':0})

In [48]:
print(f'Null Values : {df.isnull().sum().sum()}')

Null Values : 0


In [49]:
df['smoking_history'].value_counts()

smoking_history
No Info        35816
never          35095
former          9352
current         9286
not current     6447
ever            4004
Name: count, dtype: int64

In [50]:
categorical_cols = ['smoking_history']

encoder = OneHotEncoder(drop='first', sparse=False)
encoded_cols = encoder.fit_transform(df[categorical_cols])
encoded_df = pd.DataFrame(encoded_cols, columns=encoder.get_feature_names_out(categorical_cols))

df = pd.concat([df.drop(categorical_cols, axis=1), encoded_df], axis=1)

In [51]:
scaling_cols = ['HbA1c_level', 'bmi', 'blood_glucose_level']
scalar = StandardScaler()
df[scaling_cols] = scalar.fit_transform(df[scaling_cols])

In [52]:
X = df.drop('diabetes', axis=1)
y = df['diabetes']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [53]:
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test.values , dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [54]:
class DiseaseData(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]    

In [55]:
train_dataset = DiseaseData(X_train_tensor, y_train_tensor)
test_dataset = DiseaseData(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [56]:
class ANNmodel(nn.Module):
  def __init__(self, input_dim):
    super().__init__()
    self.container = nn.Sequential(
        nn.Linear(input_dim, 1024),
        nn.ReLU(),
        nn.Linear(1024, 512),
        nn.ReLU(),
        nn.Linear(512, 64),
        nn.ReLU(),
        nn.Linear(64, 1)
    )

  def forward(self, X):
    return self.container(X)  

In [57]:
model = ANNmodel(input_dim=X_train_tensor.shape[1])
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [58]:
num_epochs = 50
best_f1 = 0
patience = 5
counter = 0

from sklearn.metrics import f1_score, classification_report

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # Validation on test set
    model.eval()
    with torch.no_grad():
        y_pred = torch.sigmoid(model(X_test_tensor))
        y_pred_class = (y_pred >= 0.5).int()
    
    f1 = f1_score(y_test_tensor, y_pred_class)
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}, F1: {f1:.4f}")
    
    # Early stopping
    if f1 > best_f1:
        best_f1 = f1
        counter = 0
        torch.save(model.state_dict(), "best_diabetes_model.pth")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered!")
            break

# Load best model
model.load_state_dict(torch.load("best_diabetes_model.pth"))

Epoch 1, Loss: 0.1550, F1: 0.7379
Epoch 2, Loss: 0.1187, F1: 0.7399
Epoch 3, Loss: 0.1049, F1: 0.7725
Epoch 4, Loss: 0.0988, F1: 0.7805
Epoch 5, Loss: 0.0925, F1: 0.7903
Epoch 6, Loss: 0.0924, F1: 0.7785
Epoch 7, Loss: 0.0907, F1: 0.7653
Epoch 8, Loss: 0.0893, F1: 0.7468
Epoch 9, Loss: 0.0879, F1: 0.8045
Epoch 10, Loss: 0.0909, F1: 0.7839
Epoch 11, Loss: 0.0881, F1: 0.7890
Epoch 12, Loss: 0.0871, F1: 0.7759
Epoch 13, Loss: 0.0896, F1: 0.7204
Epoch 14, Loss: 0.0864, F1: 0.8045
Early stopping triggered!


<All keys matched successfully>

In [59]:
y_pred_prob = torch.sigmoid(model(X_test_tensor))
y_pred_class = (y_pred_prob >= 0.3).int()
print(classification_report(y_test_tensor, y_pred_class))

              precision    recall  f1-score   support

         0.0       0.98      0.99      0.98     18300
         1.0       0.84      0.74      0.79      1700

    accuracy                           0.97     20000
   macro avg       0.91      0.87      0.89     20000
weighted avg       0.97      0.97      0.97     20000



In [62]:
import joblib

joblib.dump(scalar, "scaler.pkl")

['scaler.pkl']